In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [5]:
from src.data_loader import load_data

(
    X_train,
    X_test,
    y_train,
    y_test,
    FEATURE_COLS
) = load_data()



Data range : 2018-05-25 → 2026-06-11
Total rows : 9910
Total Features : 41
Features       : ['rsi', 'macd', 'macd_signal', 'macd_hist', 'ema_20', 'ema_50', 'ema_cross', 'bb_upper', 'bb_lower', 'bb_width', 'atr', 'volume_ma', 'volume_ratio', 'return_1d', 'return_5d', 'return_10d', 'lag_1', 'lag_2', 'lag_3', 'cdl_doji', 'cdl_hammer', 'cdl_engulfing', 'cdl_shooting_star', 'cdl_morning_star', 'cdl_evening_star', 'cdl_marubozu', 'cdl_3white_soldiers', 'pattern_strength', 'pattern_count', 'nifty_return_1d', 'nifty_return_5d', 'nifty_rsi', 'nifty_above_ema', 'rel_strength_1d', 'rel_strength_5d', 'rsi_vs_nifty', 'ticker_HDFCBANK.NS', 'ticker_INFY.NS', 'ticker_RELIANCE.NS', 'ticker_SBIN.NS', 'ticker_TCS.NS']

Train : 7430 rows | 2018-05-25 → 2024-06-07
Test  : 2480 rows  | 2024-06-10 → 2026-06-11

Train UP% : 55.1%
Test  UP% : 50.6%

Scaler and feature_cols saved


In [6]:
import optuna
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

tscv = TimeSeriesSplit(
    n_splits=7
)

In [7]:
def objective_rf(trial):

    params = {

        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                100,
                500
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                3,
                15
            ),

        "min_samples_split":
            trial.suggest_int(
                "min_samples_split",
                2,
                30
            ),

        "min_samples_leaf":
            trial.suggest_int(
                "min_samples_leaf",
                1,
                15
            ),

        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestClassifier(
        **params
    )

    scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(
            X_tr,
            y_tr
        )

        probs = model.predict_proba(
            X_val
        )[:, 1]

        score = roc_auc_score(
            y_val,
            probs
        )

        scores.append(score)

    return np.mean(scores)

In [8]:
rf_study = optuna.create_study(
    direction="maximize"
)

rf_study.optimize(
    objective_rf,
    n_trials=30,
    show_progress_bar=True
)


print("Best ROC-AUC:")
print(rf_study.best_value)

print("\nBest Params:")
print(rf_study.best_params)

[I 2026-06-12 17:09:09,630] A new study created in memory with name: no-name-5bc17439-f043-4851-85b8-4b96c079fdaa


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-12 17:09:12,892] Trial 0 finished with value: 0.5341089961741943 and parameters: {'n_estimators': 199, 'max_depth': 11, 'min_samples_split': 13, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.5341089961741943.
[I 2026-06-12 17:09:19,229] Trial 1 finished with value: 0.5296010440447994 and parameters: {'n_estimators': 403, 'max_depth': 11, 'min_samples_split': 26, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.5341089961741943.
[I 2026-06-12 17:09:22,124] Trial 2 finished with value: 0.5262910014694794 and parameters: {'n_estimators': 202, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.5341089961741943.
[I 2026-06-12 17:09:26,913] Trial 3 finished with value: 0.5312154764084454 and parameters: {'n_estimators': 472, 'max_depth': 3, 'min_samples_split': 23, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.5341089961741943.
[I 2026-06-12 17:09:32,920] Trial 4 finished with value: 0.5303071362770978 and parameters

In [11]:
from xgboost import XGBClassifier
def objective_xgb(trial):

    params = {

        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                100,
                600
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                3,
                8
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.3,
                log=True
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

        "min_child_weight":
            trial.suggest_int(
                "min_child_weight",
                1,
                10
            ),

        "gamma":
            trial.suggest_float(
                "gamma",
                0,
                5
            ),

        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        probs = model.predict_proba(X_val)[:, 1]

        scores.append(
            roc_auc_score(
                y_val,
                probs
            )
        )

    return np.mean(scores)

In [12]:
xgb_study = optuna.create_study(
    direction="maximize"
)

xgb_study.optimize(
    objective_xgb,
    n_trials=50,
    show_progress_bar=True
)

print(xgb_study.best_value)
print(xgb_study.best_params)

[I 2026-06-12 17:18:01,047] A new study created in memory with name: no-name-66ee1b63-850e-47b6-84fa-449ee0a7eecd


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-12 17:18:04,164] Trial 0 finished with value: 0.5291690410704106 and parameters: {'n_estimators': 447, 'max_depth': 6, 'learning_rate': 0.10184752782838195, 'subsample': 0.9436563883776519, 'colsample_bytree': 0.9462814998573569, 'min_child_weight': 3, 'gamma': 3.2781209730870007}. Best is trial 0 with value: 0.5291690410704106.
[I 2026-06-12 17:18:05,636] Trial 1 finished with value: 0.5321723707749095 and parameters: {'n_estimators': 138, 'max_depth': 4, 'learning_rate': 0.013695546860000829, 'subsample': 0.9519390585264315, 'colsample_bytree': 0.7404537782857342, 'min_child_weight': 3, 'gamma': 2.809946620649699}. Best is trial 1 with value: 0.5321723707749095.
[I 2026-06-12 17:18:09,193] Trial 2 finished with value: 0.543020345645212 and parameters: {'n_estimators': 295, 'max_depth': 6, 'learning_rate': 0.05831129362177304, 'subsample': 0.8973138266939267, 'colsample_bytree': 0.6167000303918856, 'min_child_weight': 8, 'gamma': 1.1555225675697827}. Best is trial 2 with va

In [15]:
from sklearn.svm import SVC
def objective_svm(trial):

    params = {

        "C": trial.suggest_float(
            "svm_C",
            0.1,
            10.0,
            log=True
        ),

        "gamma": trial.suggest_float(
            "svm_gamma",
            0.001,
            0.1,
            log=True
        ),

        "kernel": "rbf",

        "probability": True,

        "random_state": 42
    }

    model = SVC(**params)

    scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        probs = model.predict_proba(X_val)[:, 1]

        score = roc_auc_score(
            y_val,
            probs
        )

        scores.append(score)

    return np.mean(scores)

In [16]:
svm_study = optuna.create_study(
    direction="maximize"
)

svm_study.optimize(
    objective_svm,
    n_trials=30,
    show_progress_bar=True
)

print("Best ROC-AUC:")
print(svm_study.best_value)

print("\nBest Params:")
print(svm_study.best_params)

[I 2026-06-12 17:30:06,714] A new study created in memory with name: no-name-ede570b9-5fe8-4fc9-aba7-f2f731ff7395


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-12 17:30:27,814] Trial 0 finished with value: 0.5280133582237815 and parameters: {'svm_C': 0.2508744932172762, 'svm_gamma': 0.001083257394242949}. Best is trial 0 with value: 0.5280133582237815.
[I 2026-06-12 17:30:49,423] Trial 1 finished with value: 0.538033251915966 and parameters: {'svm_C': 6.21787223090306, 'svm_gamma': 0.00268972229549368}. Best is trial 1 with value: 0.538033251915966.
[I 2026-06-12 17:31:10,157] Trial 2 finished with value: 0.5328087652184604 and parameters: {'svm_C': 0.8516124723353015, 'svm_gamma': 0.004924181534647826}. Best is trial 1 with value: 0.538033251915966.
[I 2026-06-12 17:31:31,601] Trial 3 finished with value: 0.5338049810114557 and parameters: {'svm_C': 4.981333357415419, 'svm_gamma': 0.0019502133635244032}. Best is trial 1 with value: 0.538033251915966.
[I 2026-06-12 17:31:52,682] Trial 4 finished with value: 0.5241642909071728 and parameters: {'svm_C': 0.1945778510641248, 'svm_gamma': 0.08653051223761117}. Best is trial 1 with value